# Terms and patient risk across clinical note types (pre-landmark text)

Companion to [13: pathology notes only](13_pathology_topics_vs_risk.ipynb). This notebook reads **Clinician**
(`PROGRESS_NOTES.parquet`, which already includes discharge summaries), **Imaging** (`IMAGING_NOTES.parquet`), and
**Pathology** (`PATHOLOGY_NOTES.parquet`). Edit `NOTE_TYPES` / `NOTE_PATHS` to choose sources. Missing selected files
stop the run rather than silently change it. Run manuscript notebook 04 first for the transferred platinum-trained
XGBoost patient risk scores.

The primary analysis relates patient TF-IDF weights to continuous raw risk, using Spearman correlation with
patient permutations and bootstrap intervals, plus adjusted linear regression. Decile term enrichment, the curated
lexicon, and binary term/lexicon associations with risk percentile are supplementary. As in notebook 13, these
analyses run on pooled notes. The shared BERTopic model and its pooled / per-type topic analyses
moved to [16_clinical_notes_bertopic_vs_risk.ipynb](16_clinical_notes_bertopic_vs_risk.ipynb) on 2026-11-18.

All finite scored patients are eligible, including middle-risk patients. No extraction manifest is used or written.
The pathology lexicon measures mentions in non-pathology
notes too; a mention is not a confirmed diagnosis, and specimen-site inference is not applied to these mixed note types.

Aggregate outputs go to `<COMPASS data root>/testing/clinical_notes_topics_vs_risk/`, with prefix `clinical_notes_`:
`tfidf_risk_association.csv` and `tfidf_risk_coverage.csv`, plus supplementary term/lexicon enrichment and score associations. Patient IDs and raw notes remain in memory only;
no report text is saved. Keep aggregate term tables protected as well.
The [Elio extraction](../../patient_extraction/01_high_vs_low_risk_transfer_pxs.ipynb) is a separate notebook.

**Timing: pre-landmark text only.** At each landmark, a note is used when its date is at or before ADT + landmark (notes before ADT start are included). Days run from ADT start (`TREATMENT_ANCHOR_DATE` in `DATA_PATH`) to the note's `EVENT_DATE`, dated as in notebook 11. Undated notes and notes dated after ADT + landmark are dropped. The window is applied per landmark, so a note at ADT + 120 d is used at +180 but not at +0 or +90. Outputs go to the `pre_landmark/` subfolder of the output folder. Every selected note type needs `EVENT_DATE`.

Associations use only notes written before the landmark, so they cannot reflect later outcomes, but they
remain descriptive and do not establish prospective predictive value.


## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
NOTES_ROOT = Path(os.environ.get("PROFILE_CLINICAL_NOTES", str(Path(cp._PROFILE_DATA_ROOT) / "CLINICAL_NOTES")))
NOTE_PATHS = {
    "Clinician": NOTES_ROOT / "PROGRESS_NOTES.parquet",
    "Imaging": NOTES_ROOT / "IMAGING_NOTES.parquet",
    "Pathology": Path(os.environ.get("PROFILE_PATHOLOGY_NOTES", str(NOTES_ROOT / "PATHOLOGY_NOTES.parquet"))),
}
NOTE_TYPES = ("Clinician", "Imaging", "Pathology")

OUTPUT_DIR = DATA_ROOT / "testing" / "clinical_notes_topics_vs_risk"

# Pre-landmark text only: at each landmark, notes with EVENT_DATE <= ADT + landmark (pre-ADT notes included).
# Undated notes are dropped. Outputs go to the pre_landmark/ subfolder.
DATA_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"   # ADT anchor dates (TREATMENT_ANCHOR_DATE)
OUTPUT_DIR = OUTPUT_DIR / "pre_landmark"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
DECILE = 0.10                      # top and bottom 10% at each landmark

# Pooled clinical-note term enrichment (top vs bottom decile)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group

# Stratified sensitivity tests: report-count band x note-type availability pattern
ADJUST_REPORT_BANDS = (1, 2, 4)    # band lower edges: 1, 2-3, 4+ reports

# Primary continuous TF-IDF/risk analysis; whole-patient resampling, separately at each landmark.
TFIDF_SETTINGS = dict(min_patients=30, min_term_patients=TERM_MIN_PATIENTS,
                      n_permutations=9999, n_bootstrap=1000, seed=42, block_size=64)


## Risk scores (all patients)

One full-model XGBoost score per patient and landmark. Invalid scores are excluded. Decile labels support the legacy
contrasts only; coincident cutoffs leave all finite scores in the middle group and do not block continuous analyses.


In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def decile_flags(g):
    if not 0 < DECILE < 0.5:
        raise ValueError("DECILE must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    lo, hi = risk[valid].quantile([DECILE, 1 - DECILE]) if valid.any() else (np.nan, np.nan)
    if not valid.any() or lo >= hi:
        print(f"Landmark {landmark}: no distinct decile cutoffs; legacy extreme-group contrasts unavailable.")
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two extreme deciles are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    if lo < hi:
        g.loc[risk >= hi, "decile_group"] = "top"
        g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: decile_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

## Pooled clinical-note term enrichment: top vs bottom decile

At each landmark, the patients in the **top** and **bottom** deciles of that landmark's score are compared (ties at a
cutoff stay in the extreme groups; patients in neither decile are not analysed). Two contrasts per landmark, each reading
"overrepresented in `group` relative to `control`": top vs bottom and bottom vs top (both directions, because the test
is one-sided). Each patient's group can differ between landmarks.

- **Reports:** the pre-landmark clinical notes of each scored patient (see the overview). Groups come from each
  landmark's score. Pre-landmark notes were already on file when the patient was scored, so terms cannot reflect later progression. Patients with no
  clinical note in the window drop out of that landmark's denominators.
- **Terms:** lowercased letter-only unigrams and bigrams (numbers, dates and IDs drop out), English stop words removed,
  counted within each report so bigrams never span two reports. A term is kept if it appears in at least
  `TERM_MIN_PATIENTS` patients at the landmark, which reduces rare strings but does not guarantee de-identification; keep outputs protected.
- **Test:** a patient "has" a term if any of their reports contains it. For each contrast, one-sided Fisher's
  exact test for overrepresentation in `group`, Benjamini-Hochberg q-values within each contrast, and the odds ratio
  with a +0.5 correction. Mean patient-level TF-IDF (sublinear term frequency over each patient's pooled counts) is reported
  for each group as a descriptive column. The `*_mh` columns repeat the test stratified by report count and note-type
  availability (Mantel-Haenszel; see below).

In [ ]:
import json
import re

import polars as pl

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"   # patients in neither decile: a label only, never analysed
CONTRASTS = (("top", "bottom"), ("bottom", "top"))   # one-sided, so both directions

def rpt_key(values):
    """RPT_ID as a string join key; whole-number IDs lose any float formatting ("12.0" -> "12")."""
    s = pd.Series(values).reset_index(drop=True)
    num = pd.to_numeric(s, errors="coerce")
    whole = num.notna() & np.isfinite(num) & (num % 1 == 0)
    key = s.astype("string").str.strip()
    key[whole] = num[whole].astype("int64").astype(str)
    return key.where(s.notna())



def note_keys(frame):
    """Collision-safe identity across sources/patients; missing report IDs have no embedding join key."""
    report = rpt_key(frame["RPT_ID"])
    patient = rpt_key(frame["DFCI_MRN"])
    kind = frame["NOTE_TYPE"].reset_index(drop=True).astype("string")
    valid = report.notna() & patient.notna() & kind.notna()
    keys = pd.Series(pd.NA, index=range(len(frame)), dtype="string")
    keys.loc[valid] = [json.dumps([t, p, r], separators=(",", ":"))
                       for t, p, r in zip(kind[valid], patient[valid], report[valid])]
    return keys


def patient_note_strata(reports):
    """Report-count band x available note types; no pathology specimen heuristics on other note types."""
    per = reports.groupby("DFCI_MRN").size()
    band = np.searchsorted(ADJUST_REPORT_BANDS, per.to_numpy(), side="right") - 1
    mask = np.zeros(len(per), dtype=int)
    for bit, kind in enumerate(NOTE_TYPES):
        ids = reports.loc[reports["NOTE_TYPE"].eq(kind), "DFCI_MRN"].unique()
        mask += per.index.isin(ids).astype(int) * (2 ** bit)
    return pd.Series(band * (2 ** len(NOTE_TYPES)) + mask, index=per.index, name="stratum")


def note_views(reports):
    """Pooled and per-type views using the same report membership and shared topic vocabulary."""
    yield "All", reports
    for kind in NOTE_TYPES:
        yield kind, reports[reports["NOTE_TYPE"].eq(kind)]


groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

if not NOTE_TYPES or len(set(NOTE_TYPES)) != len(NOTE_TYPES) or "All" in NOTE_TYPES:
    raise ValueError("Select unique NOTE_TYPES; All is reserved for pooled results.")
scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
frames = []
for kind in NOTE_TYPES:
    path = Path(NOTE_PATHS[kind])
    if not path.exists():
        raise FileNotFoundError(f"Selected {kind} notes not found: {path}; edit NOTE_TYPES/NOTE_PATHS if intentional.")
    scan = pl.scan_parquet(path)
    schema = scan.collect_schema().names()
    required = {"DFCI_MRN", "RPT_TEXT"}
    if not required.issubset(schema):
        raise ValueError(f"{kind} notes missing columns: {sorted(required - set(schema))}")
    cols = [c for c in ("DFCI_MRN", "RPT_TEXT", "RPT_ID", "EVENT_DATE", "PROC_DESC") if c in schema]
    frame = (scan.select(cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids)).collect().to_pandas())
    if "RPT_ID" not in frame:
        frame["RPT_ID"] = pd.NA   # still usable for terms, but cannot join embeddings
    if "EVENT_DATE" not in frame:
        raise ValueError(f"{kind} notes have no EVENT_DATE; the pre-landmark window needs dated notes.")
    frame["NOTE_TYPE"] = kind
    frames.append(frame)
all_notes = pd.concat(frames, ignore_index=True)
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes["rpt_key"] = note_keys(all_notes).to_numpy()
all_notes["note_date"] = pd.to_datetime(
    all_notes["EVENT_DATE"].astype("string").str.extract(r"^(\d{4}-\d{2}-\d{2})", expand=False), errors="coerce")
identified = all_notes[all_notes["rpt_key"].notna()]
if identified.groupby("rpt_key")["RPT_TEXT"].nunique().gt(1).any():
    raise ValueError("Conflicting note text for the same note-type/patient/report identity.")
# Reconcile dates before keeping one copy: an undated copy must not hide a dated report.
dates = identified.groupby("rpt_key")["note_date"]
if dates.nunique().gt(1).any():
    raise ValueError("Conflicting note dates for the same note-type/patient/report identity.")
all_notes.loc[identified.index, "note_date"] = dates.transform("first")
all_notes = all_notes[all_notes["rpt_key"].isna() | ~all_notes["rpt_key"].duplicated()].reset_index(drop=True)
all_notes["note_row"] = np.arange(len(all_notes))

# --- Pre-landmark window. Days from ADT start = EVENT_DATE (its YYYY-MM-DD prefix) - TREATMENT_ANCHOR_DATE,
# dated as in notebook 11. Undated notes are dropped.
anchor_cols = ["DFCI_MRN", "TREATMENT_ANCHOR_DATE"]
anchor = (pd.read_parquet(DATA_PATH, columns=anchor_cols) if str(DATA_PATH).endswith((".parquet", ".pq"))
          else pd.read_csv(DATA_PATH, usecols=anchor_cols, low_memory=False)).dropna().drop_duplicates()
anchor["TREATMENT_ANCHOR_DATE"] = pd.to_datetime(anchor["TREATMENT_ANCHOR_DATE"], errors="coerce", format="mixed")
anchor = anchor.dropna().drop_duplicates()
anchor["DFCI_MRN"] = pd.to_numeric(anchor["DFCI_MRN"], errors="raise").astype("int64")
if anchor["DFCI_MRN"].duplicated().any():
    raise ValueError("More than one TREATMENT_ANCHOR_DATE for some patients.")
anchor = anchor.set_index("DFCI_MRN")["TREATMENT_ANCHOR_DATE"]
n_missing = sum(i not in anchor.index for i in scored_ids)
if n_missing:
    raise ValueError(f"{n_missing} scored patients lack an ADT anchor in DATA_PATH; reconcile it with the scores.")
note_date = all_notes["note_date"]
all_notes["days_from_adt"] = (note_date - all_notes["DFCI_MRN"].map(anchor)).dt.days.astype(float).to_numpy()


def in_window(reports, lm):
    """Rows of `reports` dated at or before ADT + `lm`, pre-ADT included (undated rows never)."""
    days = reports["days_from_adt"]
    return (days.le(lm)).fillna(False).to_numpy(dtype=bool)


def window_summary(rows):
    """Prints, per landmark, how many of the scored patients' notes are pre-landmark (aggregate counts only)."""
    print("Pre-landmark notes (note date <= ADT + landmark; undated notes dropped):")
    for r in rows:
        print(f"  landmark {r['landmark']}: {r['in_window']:,} of {r['any_date']:,} notes "
              f"({r['undated']:,} undated), {r['patients_in_window']:,} patients")


# The pre-landmark notes of the patients scored at each landmark.
reports_by_lm, strata_by_lm, size_rows, window_rows = {}, {}, [], []
for lm, grp in groups_by_lm.items():
    pool = all_notes[all_notes["DFCI_MRN"].isin(grp.index)]
    rep_lm = pool[in_window(pool, lm)].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    window_rows.append(dict(landmark=lm, any_date=len(pool), in_window=len(rep_lm),
                            undated=int(pool["days_from_adt"].isna().sum()),
                            patients_in_window=int(rep_lm["DFCI_MRN"].nunique())))
    strata_by_lm[lm] = patient_note_strata(rep_lm)
    for kind, view in note_views(rep_lm):
        per = view.groupby("DFCI_MRN").size()
        for name in TERM_GROUPS:
            members = grp.index[grp.eq(name).to_numpy()]
            counts = per[per.index.isin(members)]
            size_rows.append(dict(landmark=lm, note_type=kind, group=name, patients_scored=len(members),
                                  patients_with_report=len(counts), reports=int(counts.sum()),
                                  median_reports=float(counts.median()) if len(counts) else np.nan))
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "note_type", "group"])
print(f"Loaded {len(all_notes):,} clinical notes across {len(NOTE_TYPES)} selected types (any date, before the window).")
window_summary(window_rows)
try:
    from IPython.display import display
except ImportError:
    display = print
display(group_sizes)


In [ ]:
from scipy import sparse
from scipy.stats import hypergeom, norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# Applied after lowercasing: letters only, >= 3 characters.
TOKEN_PATTERN = r"(?u)\b[a-z][a-z]{2,}\b"
MH_COLS = ["odds_ratio_mh", "p_value_mh", "q_value_mh"]
ENRICHMENT_COLS = ["landmark", "group", "control", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS,
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values over the finite p values; NaN stays NaN (cluster scipy predates
    false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if ok.size:
        order = ok[np.argsort(p[ok])]
        ranked = p[order] * ok.size / np.arange(1, ok.size + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient.
    """
    text = reports["RPT_TEXT"].to_numpy()
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(text)
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def colsum(m):
    return np.asarray(m.sum(axis=0)).ravel().astype(float)


def mantel_haenszel(present, in_group, control, strata):
    """One-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio per feature column, across strata.

    A stratum without patients from both sides carries no information and is skipped; if none is left, both are NaN.
    The odds ratio adds 0.5 to every cell of a stratum that has an empty cell.
    """
    n_feat = present.shape[1]
    A, E, V, R, S = (np.zeros(n_feat) for _ in range(5))
    informative = False
    for k in pd.unique(strata[in_group | control]):
        g, c = in_group & (strata == k), control & (strata == k)
        n1, n0 = int(g.sum()), int(c.sum())
        if not (n1 and n0):
            continue
        informative = True
        n = n1 + n0
        a, cc = colsum(present[g]), colsum(present[c])
        b, d, m = n1 - a, n0 - cc, a + cc
        A += a
        E += n1 * m / n
        V += n1 * n0 * m * (n - m) / (n ** 2 * (n - 1))
        h = 0.5 * ((a == 0) | (b == 0) | (cc == 0) | (d == 0))
        R += (a + h) * (d + h) / (n + 4 * h)
        S += (b + h) * (cc + h) / (n + 4 * h)
    if not informative:
        return np.full(n_feat, np.nan), np.full(n_feat, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(V > 0, norm.sf((A - E - 0.5) / np.sqrt(V)), 1.0)
    return R / S, p


def fisher_enrichment(present, label, features, landmark, strata=None):
    """One-sided Fisher overrepresentation of each feature column for each (group, control) in CONTRASTS.

    present: patients x features 0/1 sparse matrix; label: each patient's group; strata (optional): each patient's
    stratum for the Mantel-Haenszel columns (NaN without it). Rows come out in CONTRASTS order, len(features) rows
    per contrast whose group and control both have patients.
    """
    out = []
    for name, ref in CONTRASTS:
        in_group, control = label == name, label == ref
        n1, n0 = int(in_group.sum()), int(control.sum())
        if n1 == 0 or n0 == 0:
            continue
        a, c = colsum(present[in_group]), colsum(present[control])
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        if strata is None:
            or_mh = p_mh = np.full(len(features), np.nan)
        else:
            or_mh, p_mh = mantel_haenszel(present, in_group, control, np.asarray(strata))
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "control": ref, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a.astype(int), "control_with_feature": c.astype(int),
            "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
            "odds_ratio_mh": or_mh, "p_value_mh": p_mh, "q_value_mh": bh_qvalues(p_mh),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def sort_contrasts(df):
    """Landmark, then CONTRASTS order, then p value and odds ratio."""
    rank = {pair: i for i, pair in enumerate(CONTRASTS)}
    order = pd.Series([rank[pair] for pair in zip(df["group"], df["control"])], index=df.index, dtype=float)
    return (df.assign(_contrast=order)
            .sort_values(["landmark", "_contrast", "p_value", "odds_ratio"], ascending=[True, True, True, False])
            .drop(columns="_contrast").reset_index(drop=True))


def contrast_title(lm, name, ref):
    return f"Landmark {lm} · {name} decile vs {ref} decile"


term_matrices = {}   # landmark -> (patient ids, terms, presence, TF-IDF); reused by the score association cell


def term_enrichment(reports, groups, landmark):
    """Fisher overrepresentation of each term for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(reports["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 report
    terms, present, tfidf = patient_term_matrices(reports, ids)
    term_matrices[landmark] = (ids, terms, present, tfidf)
    if not len(terms):
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    label = groups.reindex(ids).to_numpy()
    res = fisher_enrichment(present, label, terms, landmark, strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    mean_tfidf = {name: np.asarray(tfidf[label == name].mean(axis=0)).ravel()
                  for name in set(res["group"]) | set(res["control"])}
    pairs = list(res[["group", "control"]].drop_duplicates().itertuples(index=False))
    res["mean_tfidf_group"] = np.concatenate([mean_tfidf[g] for g, _ in pairs])
    res["mean_tfidf_control"] = np.concatenate([mean_tfidf[c] for _, c in pairs])
    return res.rename(columns={"feature": "term", "group_with_feature": "group_with_term",
                               "control_with_feature": "control_with_term"})


enrichment = pd.concat(
    [term_enrichment(reports_by_lm[lm], groups_by_lm[lm], lm) for lm in LANDMARKS if lm in reports_by_lm],
    ignore_index=True,
).reindex(columns=ENRICHMENT_COLS)
enrichment = sort_contrasts(enrichment)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enrichment_path = OUTPUT_DIR / "clinical_notes_term_enrichment.csv"
enrichment.to_csv(enrichment_path, index=False)
print(f"wrote {enrichment_path} ({len(enrichment):,} term x contrast rows)")


## Primary analysis: continuous TF-IDF and patient risk

Every eligible patient contributes their continuous raw risk score and one TF-IDF vector; decile labels do not
enter this analysis. We reuse the vocabulary and patient-level TF-IDF fitted separately within each landmark.
Weights are zero when the term is absent. Positive effects indicate increasing relative term prominence with risk;
these weights are not absolute mention counts or measures of disease severity.

- **Primary: Spearman rho.** A rank correlation between raw risk and each term's TF-IDF weight. Ties, including
  zero weights, receive average ranks. Two-sided patient permutations test an independence null by shuffling scores
  between patients, with each patient's complete vector kept together. This is an unadjusted descriptive association.
  Paired patient bootstrap samples are re-ranked to obtain percentile 95% intervals, conditional on the fitted
  vocabulary and IDF. Constant bootstrap samples are omitted; intervals are withheld if fewer than 80% of samples
  are usable. `n_bootstrap_valid` and `ci_status` report this explicitly.
- **Sensitivity: adjusted linear regression.** TF-IDF weight is the response; raw risk is standardized across eligible
  patients within the landmark. `beta_per_sd` is the change in mean weight per one SD higher risk. Adjustment includes log(1 + report count) and patient note-type fractions, omitting the first sorted type as reference.
  HC3 heteroskedasticity-robust standard errors and residual-df t inference provide p-values and 95% intervals.
  Constant covariates are omitted; collinear or unstable designs are marked as skipped. This is a linear descriptive
  sensitivity analysis; adjustment does not establish causality.
- **Inference and coverage.** Require `TFIDF_SETTINGS['min_patients']` patients and
  `TFIDF_SETTINGS['min_term_patients']` patients mentioning a term. Invalid scores are excluded; constant scores
  or term weights are skipped. BH correction covers all tested terms and landmarks within each method, separately
  in each notebook/timing folder. Landmarks share patients and do not constitute independent replication.
  The minimum permutation p-value is `1 / (n_permutations + 1)`; increase permutations if this resolution limits FDR
  discoveries. Term blocks bound memory use; all blocks use the same patient resampling draws.

Aggregate effects go to `clinical_notes_tfidf_risk_association.csv`, with one row per term, landmark and method;
`clinical_notes_tfidf_risk_coverage.csv` records patient and vocabulary coverage. No IDs or raw text are exported.
Decile enrichment and binary term/lexicon associations below remain supplementary analyses.


In [ ]:
from COMPASS.multivariable_models import text_tfidf_association as tfa

try:
    from IPython.display import display
except ImportError:
    display = print

tfidf_frames, tfidf_coverage = [], []
for lm in LANDMARKS:
    g = by_lm[lm]
    risk = pd.Series(g["risk_score"].to_numpy(dtype=float),
                     index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="raise").astype("Int64")))
    entry = term_matrices.get(lm)
    tfidf_coverage.append(dict(landmark=lm, patients_scored=int(np.isfinite(risk).sum()),
        patients_with_report=reports_by_lm[lm]["DFCI_MRN"].nunique(),
        n_terms=0 if entry is None else len(entry[1])))
    if entry is None or not len(entry[1]):
        continue
    ids, terms, _, tfidf = entry
    covariates = tfa.patient_covariates(reports_by_lm[lm], ids, scope="clinical_notes")
    tfidf_frames.append(tfa.continuous_tfidf_association(
        tfidf, ids, terms, risk, covariates, lm, **TFIDF_SETTINGS))

tfidf_association = tfa.adjust_fdr(tfidf_frames)
tfidf_association.to_csv(OUTPUT_DIR / "clinical_notes_tfidf_risk_association.csv", index=False)
pd.DataFrame(tfidf_coverage).to_csv(OUTPUT_DIR / "clinical_notes_tfidf_risk_coverage.csv", index=False)
print(f"Continuous TF-IDF/risk: {len(tfidf_association):,} term x landmark x method rows; "
      f"permutation p-value resolution {1 / (TFIDF_SETTINGS['n_permutations'] + 1):.3g}")
display(pd.DataFrame(tfidf_coverage))
for method in ("spearman", "ols_adjusted"):
    sub = tfidf_association[tfidf_association["method"].eq(method)]
    print(f"{method}: {sub['status'].eq('ok').sum():,} estimated; "
          f"{sub['q_value'].lt(TERM_FDR).sum():,} at q < {TERM_FDR} across terms and landmarks")
    effect = "spearman_rho" if method == "spearman" else "beta_per_sd"
    for direction, mask in (("higher", sub[effect].gt(0)), ("lower", sub[effect].lt(0))):
        print(f"  Terms with {direction} prominence as risk increases:")
        display(sub[mask].sort_values("p_value")[
            ["landmark", "term", "n_patients", "prevalence", effect, "ci95_low", "ci95_high",
             "p_value", "q_value", "ci_status"]].head(TERM_TOP_N).round(4).reset_index(drop=True))
    if sub.empty:
        print("  No retained vocabulary to test.")
    elif sub['status'].ne('ok').any():
        display(sub.loc[sub['status'].ne('ok'), 'status'].value_counts().rename("skipped_terms"))


In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

show_cols = ["term", "group_with_term", "n_group", "pct_group", "control_with_term", "n_control", "pct_control",
             "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh", "mean_tfidf_group", "mean_tfidf_control"]
for lm in LANDMARKS:
    for name, ref in CONTRASTS:
        sub = enrichment[(enrichment["landmark"] == lm) & (enrichment["group"] == name) & (enrichment["control"] == ref)]
        title = contrast_title(lm, name, ref)
        if sub.empty:
            print(f"\n{title}: no terms (too few patients or reports)")
            continue
        sig = sub[sub["q_value"] < TERM_FDR]
        note = f"q < {TERM_FDR}: {len(sig):,} terms"
        if sig.empty:
            sig, note = sub, f"none at q < {TERM_FDR}; top {TERM_TOP_N} by p shown"
        top = sig.sort_values(["p_value", "odds_ratio"], ascending=[True, False]).head(TERM_TOP_N)
        print(f"\n{title} ({note})")
        display(top[show_cols].round(3).reset_index(drop=True))

## Sensitivity and complementary analyses

Pooled selected note types; same groups, landmarks and pre-landmark notes as the term enrichment. Each one addresses a specific way the term
tables can mislead.

1. **Report count and note-type availability (`*_mh` columns).** Each patient's stratum is their report-count
   band (`ADJUST_REPORT_BANDS`) crossed with which selected note types they have. Here every analysis uses all
   reports in the window (notebook 16's topic contrasts recompute it within each embedded-note subset). The term and lexicon
   tables add a one-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio across strata.
   These are NaN when no stratum contains patients from both sides of the contrast. No specimen-site heuristic
   is applied to clinician or imaging text.
2. **Curated lexicon (`clinical_notes_lexicon_enrichment.csv`).** The neuroendocrine patterns are copied from the NEPC
   annotation pipeline's `ne_gate.py`, joined by individual NE and prostate IHC markers, histologic variants, invasion
   / extension terms and the highest Gleason sum (grade group mapped to a sum). Same Fisher and MH tests.
3. **Continuous risk (`clinical_notes_score_association.csv`).** Deciles discard most of the score. For every term and
   lexicon feature, a two-sided Mann-Whitney test of the risk percentile among all scored patients with a
   report, feature present vs absent. `auc` = P(a patient with the feature ranks higher than one without). The van
   Elteren test (`*_adj`) stratifies it like item 1.

All outputs are aggregate. No MRN or report text is printed or written by these cells.

In [ ]:
# Curated lexicon, matched on lowercased report text. The NE patterns are copied verbatim (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e); update them by hand if that file changes.
NE_TERM = (r"\b(?:neuro[\s-]?endocrine|t-?nepc|nepc|small[\s-]?cell|scpc|scnc|"
           r"oat[\s-]?cell)\b")
SMALL_CELL_TERM = r"\b(?:small[\s-]?cell|scpc|scnc|oat[\s-]?cell)\b"
NE_DIAGNOSIS = (r"\b(?:t-?nepc|nepc|scpc|scnc)\b|"
                r"\b(?:neuro[\s-]?endocrine|small[\s-]?cell|oat[\s-]?cell)"
                r"(?:\s+[\w/()-]+){0,4}?\s+"
                r"(?:carcinoma|cancer|tumou?r|neoplasm|malignancy|component|histology|"
                r"variant|subtype|type|disease|prostate)\b|"
                r"\b(?:carcinoma|cancer)\s+(?:with|of)\s+(?:\w+\s+){0,2}?"
                r"(?:neuro[\s-]?endocrine|small[\s-]?cell)\b")
NE_MARKER = (r"\b(?:synaptophysin|chromogranin(?:\s+a)?|cd\s?56|insm-?1|"
             r"neuron[\s-]+specific\s+enolase|nse)\b")
NON_PROSTATE_NE = (r"\b(?:sclc|small[\s-]?cell\s+lung|merkel|carcinoid|"
                   r"pulmonary\s+(?:origin|primary|small)|lung\s+(?:origin|primary)|"
                   r"urothelial|bladder\s+(?:primary|origin|cancer|carcinoma|tumou?r)|"
                   r"pancreatic\s+(?:net|neuroendocrine)|pnet|gep[\s-]?nets?|"
                   r"well[\s-]differentiated\s+neuroendocrine\s+tumou?r)\b")
NON_SMALL_CELL = r"\bnon[\s-]*$"
# (end of the ne_gate.py copy)

LEXICON = {
    # neuroendocrine (ne_gate.py), then each NE marker from NE_MARKER on its own
    "NE term (any)": NE_TERM,
    "NE diagnosis phrase": NE_DIAGNOSIS,
    "small cell": SMALL_CELL_TERM,
    "NE marker (any)": NE_MARKER,
    "non-prostate NE primary": NON_PROSTATE_NE,
    "synaptophysin": r"\bsynaptophysin\b",
    "chromogranin": r"\bchromogranin\b",
    "CD56": r"\bcd\s?56\b",
    "INSM1": r"\binsm-?1\b",
    "NSE": r"\b(?:neuron[\s-]+specific\s+enolase|nse)\b",
    # other IHC
    "Ki-67 / MIB-1": r"\b(?:ki-?67|mib-?1)\b",
    "AR": r"\b(?:androgen\s+receptor|ar)\b",
    "NKX3.1": r"\bnkx\s?3[.\s-]?1\b",
    "PSMA": r"\b(?:psma|prostate[\s-]specific\s+membrane\s+antigen)\b",
    "ERG": r"\berg\b",
    "PTEN": r"\bpten\b",
    # histology and staging
    "ductal": r"\bductal\b",
    "intraductal": r"\bintraductal\b|\bidc-?p\b",
    "cribriform": r"\bcribriform\b",
    "sarcomatoid": r"\bsarcomatoid\b",
    "treatment effect": r"\b(?:treatment|therapy|hormon\w*|androgen[\s-]deprivation)[\s-]+(?:related\s+)?effects?\b",
    "perineural invasion": r"\bperineural\s+(?:invasion|infiltration)\b|\bpni\b",
    "lymphovascular invasion": (r"\b(?:lympho-?vascular|angiolymphatic|lymphatic|vascular)\s+(?:space\s+)?invasion\b"
                                r"|\blvi\b"),
    "extraprostatic extension": r"\bextra-?prostatic\s+extension\b|\bepe\b",
    "seminal vesicle invasion": r"\bseminal\s+vesicles?\s+(?:invasion|involvement|involved)\b|\bsvi\b",
}
LEXICON_RE = {name: re.compile(rx) for name, rx in LEXICON.items()}
NON_PREFIX_RE = re.compile(NON_SMALL_CELL)
# Gleason: the highest primary + secondary sum in any report (grade group mapped to a sum).
GLEASON_RE = re.compile(r"\bgleason\b[^.;\n]{0,40}?\b([1-5])\s*\+\s*([1-5])\b")
GRADE_GROUP_RE = re.compile(r"\b(?:grade\s+group|isup\s+grade(?:\s+group)?|gg)\s*:?\s*([1-5])\b")
GRADE_GROUP_SUM = {1: 6, 2: 7, 3: 7, 4: 8, 5: 9}
GLEASON_BINS = (("Gleason max 2-6", 2, 6), ("Gleason max 7", 7, 7), ("Gleason max 8-10", 8, 10))

MENTION_FEATURES = list(LEXICON)
LEXICON_FEATURES = MENTION_FEATURES + [label for label, _, _ in GLEASON_BINS]
LEXICON_COLS = ["landmark", "group", "control", "feature", "n_group", "n_control", "group_with_feature",
                "control_with_feature", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def lexicon_hits(text):
    """One report's lexicon features: {feature: True} for each mention, plus gleason_max.

    A match right after "non-" ("non-small cell") is not a mention.
    """
    text = text.lower()
    hits = {}
    for name, rx in LEXICON_RE.items():
        if any(not NON_PREFIX_RE.search(text[max(0, m.start() - 8):m.start()]) for m in rx.finditer(text)):
            hits[name] = True
    sums = ([int(a) + int(b) for a, b in GLEASON_RE.findall(text)]
            + [GRADE_GROUP_SUM[int(g)] for g in GRADE_GROUP_RE.findall(text)])
    if sums:
        hits["gleason_max"] = max(sums)
    return hits


# Report-level hits, once for every report at any landmark.
report_rows = np.unique(np.concatenate([rep["note_row"].to_numpy() for rep in reports_by_lm.values()]
                                       + [np.array([], dtype=int)])).astype(int)
report_lex = pd.DataFrame([lexicon_hits(t) for t in all_notes["RPT_TEXT"].to_numpy()[report_rows]],
                          index=report_rows).reindex(columns=MENTION_FEATURES + ["gleason_max"])
report_mentions = report_lex[MENTION_FEATURES].notna().to_numpy()
report_gleason = pd.to_numeric(report_lex["gleason_max"], errors="coerce").to_numpy(dtype=float)
lexicon_matrices = {}   # landmark -> (patient ids, presence); reused by the score association cell
lexicon_frames = []
for lm in LANDMARKS:
    if lm not in reports_by_lm or reports_by_lm[lm].empty:
        continue
    rep = reports_by_lm[lm]
    ids = pd.Index(sorted(rep["DFCI_MRN"].unique()), dtype="Int64")
    rows = ids.get_indexer(rep["DFCI_MRN"])
    pos = np.searchsorted(report_rows, rep["note_row"].to_numpy())
    mentions = np.zeros((len(ids), len(MENTION_FEATURES)), dtype=bool)
    np.logical_or.at(mentions, rows, report_mentions[pos])
    gleason = np.full(len(ids), np.nan)
    np.fmax.at(gleason, rows, report_gleason[pos])
    bins = np.column_stack([(gleason >= lo) & (gleason <= hi) for _, lo, hi in GLEASON_BINS])
    present = sparse.csr_matrix(np.hstack([mentions, bins]).astype(np.int64))
    lexicon_matrices[lm] = (ids, present)
    # Only features some patient has at this landmark are tested, so absent ones do not dilute the BH correction.
    seen = np.flatnonzero(colsum(present) > 0)
    lexicon_frames.append(fisher_enrichment(present[:, seen], groups_by_lm[lm].reindex(ids).to_numpy(),
                                            np.array(LEXICON_FEATURES, dtype=object)[seen], lm,
                                            strata_by_lm[lm].reindex(ids).to_numpy()))

# Rows stay in landmark, CONTRASTS, then LEXICON_FEATURES order.
lexicon_enrich = (pd.concat(lexicon_frames, ignore_index=True) if lexicon_frames else pd.DataFrame()
                  ).reindex(columns=LEXICON_COLS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lexicon_path = OUTPUT_DIR / "clinical_notes_lexicon_enrichment.csv"
lexicon_enrich.to_csv(lexicon_path, index=False)
print(f"wrote {lexicon_path} ({len(lexicon_enrich):,} feature x contrast rows)")

# One wide table per landmark: % of patients with each feature by group, then q values for every contrast
# (q = Fisher, q_mh = stratified Mantel-Haenszel). Features no patient has at that landmark are not tested.
for lm in LANDMARKS:
    sub = lexicon_enrich[lexicon_enrich["landmark"] == lm]
    if sub.empty:
        print(f"\nLandmark {lm} · lexicon: no patients with clinical notes")
        continue
    wide = pd.DataFrame(index=pd.Index(LEXICON_FEATURES, name="feature"))
    for name, ref in CONTRASTS:
        c = sub[(sub["group"] == name) & (sub["control"] == ref)].set_index("feature")
        if c.empty:
            continue
        wide[f"% {name}"], wide[f"% {ref}"] = c["pct_group"] * 100, c["pct_control"] * 100
        wide[f"q {name}>{ref}"], wide[f"q_mh {name}>{ref}"] = c["q_value"], c["q_value_mh"]
    pct_cols = [f"% {g}" for g in TERM_GROUPS if f"% {g}" in wide]
    q_cols = [col for prefix in ("q ", "q_mh ") for col in wide.columns if col.startswith(prefix)]
    wide = wide[pct_cols + q_cols]
    print(f"\nLandmark {lm} · curated lexicon (% of patients with a mention in any report)")
    display(wide.dropna(how="all").round(3))


In [ ]:
from scipy.stats import rankdata

ASSOC_COLS = ["landmark", "feature_set", "feature", "n_patients", "n_with_feature", "pct_with_feature",
              "mean_percentile_with", "mean_percentile_without", "auc", "p_value", "q_value",
              "auc_adj", "p_value_adj", "q_value_adj"]


def _mann_whitney_parts(present, score):
    """Per feature column: n with it, U of the with-feature patients, and the tie-corrected null variance of U."""
    n = present.shape[0]
    ranks = rankdata(score)
    n1 = colsum(present)
    u = np.asarray(present.T @ ranks).ravel() - n1 * (n1 + 1) / 2
    _, t = np.unique(score, return_counts=True)
    ties = float((t.astype(float) ** 3 - t).sum()) / (n * (n - 1)) if n > 1 else 0.0
    var = n1 * (n - n1) / 12 * ((n + 1) - ties)
    return n1, u, var


def score_association(present, features, score, strata, landmark, feature_set):
    """Two-sided Mann-Whitney of the risk percentile, feature present vs absent, among all scored patients with
    clinical notes; and the van Elteren test stratified like the Mantel-Haenszel columns.

    auc = P(percentile of a patient with the feature > one without); auc_adj pools the same comparison within strata.
    """
    present = present.tocsr().astype(float)
    n = present.shape[0]
    n1, u, var = _mann_whitney_parts(present, score)
    n0 = n - n1
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.maximum(np.abs(u - n1 * n0 / 2) - 0.5, 0) / np.sqrt(var)
        p = np.where(var > 0, 2 * norm.sf(z), np.nan)
        sum_with = np.asarray(present.T @ score).ravel()
        res = pd.DataFrame({
            "landmark": landmark, "feature_set": feature_set, "feature": features, "n_patients": n,
            "n_with_feature": n1.astype(int), "pct_with_feature": n1 / n,
            "mean_percentile_with": sum_with / n1, "mean_percentile_without": (score.sum() - sum_with) / n0,
            "auc": u / (n1 * n0), "p_value": p})
    # van Elteren: within-stratum U, strata weighted 1 / (N_k + 1).
    num, den, u_pool, pairs = (np.zeros(len(features)) for _ in range(4))
    for k in pd.unique(strata):
        rows = np.flatnonzero(strata == k)
        if rows.size < 2:
            continue
        n1k, uk, vk = _mann_whitney_parts(present[rows], score[rows])
        w = 1 / (rows.size + 1)
        num += w * (uk - n1k * (rows.size - n1k) / 2)
        den += w ** 2 * vk
        u_pool += uk
        pairs += n1k * (rows.size - n1k)
    with np.errstate(divide="ignore", invalid="ignore"):
        res["auc_adj"] = np.where(pairs > 0, u_pool / pairs, np.nan)
        res["p_value_adj"] = np.where(den > 0, 2 * norm.sf(np.abs(num) / np.sqrt(den)), np.nan)
    res = res[(res["n_with_feature"] > 0) & (res["n_with_feature"] < n)].reset_index(drop=True)
    res["q_value"], res["q_value_adj"] = bh_qvalues(res["p_value"]), bh_qvalues(res["p_value_adj"])
    return res[ASSOC_COLS]


assoc_frames = []
for lm in LANDMARKS:
    if lm not in by_lm:
        continue
    g = by_lm[lm]
    pct = pd.Series(g["risk_percentile"].to_numpy(dtype=float),
                    index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
    sets = []
    if lm in term_matrices and len(term_matrices[lm][1]):
        ids, terms, present, _ = term_matrices[lm]
        sets.append(("term", ids, present, terms))
    if lm in lexicon_matrices:
        ids, present = lexicon_matrices[lm]
        sets.append(("lexicon", ids, present, np.array(LEXICON_FEATURES, dtype=object)))
    for feature_set, ids, present, names in sets:
        if len(ids) < 2:
            continue
        assoc_frames.append(score_association(present, names, pct.reindex(ids).to_numpy(dtype=float),
                                              strata_by_lm[lm].reindex(ids).to_numpy(), lm, feature_set))

assoc = (pd.concat(assoc_frames, ignore_index=True) if assoc_frames else pd.DataFrame()).reindex(columns=ASSOC_COLS)
assoc_path = OUTPUT_DIR / "clinical_notes_score_association.csv"
assoc.to_csv(assoc_path, index=False)
print(f"wrote {assoc_path} ({len(assoc):,} feature x landmark rows)")

assoc_show = ["feature", "n_with_feature", "pct_with_feature", "mean_percentile_with", "mean_percentile_without",
              "auc", "q_value", "auc_adj", "q_value_adj"]
for lm in LANDMARKS:
    for feature_set in ("term", "lexicon"):
        sub = assoc[(assoc["landmark"] == lm) & (assoc["feature_set"] == feature_set)]
        if sub.empty:
            print(f"\nLandmark {lm} · {feature_set}s vs risk percentile: none tested")
            continue
        for direction, side in (("higher", sub["auc"] > 0.5), ("lower", sub["auc"] < 0.5)):
            hits = sub[side].sort_values("p_value")
            print(f"\nLandmark {lm} · {feature_set}s with {direction} risk percentiles "
                  f"(q < {TERM_FDR}: {int((hits['q_value'] < TERM_FDR).sum()):,}; top {TERM_TOP_N} by p)")
            display(hits[assoc_show].head(TERM_TOP_N).round(3).reset_index(drop=True))
